# Importing data

Before running, make sure the current shared folder is located in your personal /content/drive/MyDrive

Datasets were exported from the Kaggle COVID-19 Open Research Dataset Challenge (CORD-19), using  the nlp_data_analysis.ipynb file in the same folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import google.generativeai as genai

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

ID_COLUMN   = "paper_id"
TEXT_COLUMN = "paper_text"

In [ ]:
#make sure shared folder is in the MyDrive folder!
CURRENT_DIR = '/content/drive/MyDrive/NLP_Project'
os.chdir(CURRENT_DIR)

In [ ]:
!pip install -q \
  "transformers>=4.46.0" \
  accelerate \
  pandas \
  tqdm \
  google-generativeai

In [ ]:
df_articles = pd.read_csv('./filtered_dataset_new.csv')
df_articles.head()

In [ ]:
df_metadata = pd.read_csv('./metadata.csv')
df_metadata.head()

In [ ]:
PATH = "./filtered_dataset_new.csv"

# Only read first 100 rows
df_subset = pd.read_csv(PATH, nrows=250)

print("Columns:", df_subset.columns.tolist())
print("Subset size:", len(df_subset))
df_subset.head()

In [ ]:
PROMPT_INSTRUCTION = (
    "Summarize this research paper with a focus on the side effects or adverse "
    "outcomes associated with COVID-19. Your summary should reflect what is known, "
    "uncertain, or disputed in the paper."
)

def build_prompt(paper_text: str) -> str:
    """
    Combine the instruction and the paper into a single prompt string.
    """
    return (
        "Instruction:\n"
        f"{PROMPT_INSTRUCTION}\n\n"
        "Paper:\n"
        f"{paper_text}\n\n"
        "Summary:"
    )

In [ ]:
example_prompt = build_prompt(str(df_subset.iloc[0][TEXT_COLUMN]))
print(example_prompt[:1500])  # printing first part to see structure

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Open models
MISTRAL_ID  = "mistralai/Mistral-7B-Instruct-v0.3"
DEEPSEEK_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
#LLAMA_ID = "meta-llama/Llama-3.1-8B-Instruct"


# --- Mistral-7B-Instruct (replaces GPT-OSS slot) ---
mistral_tokenizer = AutoTokenizer.from_pretrained(MISTRAL_ID)
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_ID,
    device_map="auto",          # use GPU
    torch_dtype=torch.bfloat16, # bf16 to save VRAM
)

# --- DeepSeek-R1-Distill-Qwen-7B ---
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK_ID)
deepseek_model = AutoModelForCausalLM.from_pretrained(
    DEEPSEEK_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# --- Llama-3.1-70B-Instruct  ---
#llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_ID)
#llama_model = AutoModelForCausalLM.from_pretrained(
#    LLAMA_ID,
#    device_map="auto",
#    torch_dtype=torch.bfloat16,
#)

In [ ]:
MAX_INPUT_TOKENS = 8000   # prompt length limit
MAX_NEW_TOKENS   = 1024
TEMPERATURE       = 0.2

def generate_summary_hf(
    model,
    tokenizer,
    paper_text: str,
    temperature: float = TEMPERATURE,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    # build the prompt from instruction + paper
    prompt = build_prompt(paper_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=False,  # deterministic
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Strip off the prompt so we only keep summary text
    if "Summary:" in generated:
        generated = generated.split("Summary:", 1)[-1].strip()
    return generated

def summarize_mistral(paper_text: str) -> str:
    return generate_summary_hf(mistral_model, mistral_tokenizer, paper_text)

def summarize_deepseek(paper_text: str) -> str:
    return generate_summary_hf(deepseek_model, deepseek_tokenizer, paper_text)

#def summarize_llama(paper_text: str) -> str:
#    return generate_summary_hf(llama_model, llama_tokenizer, paper_text)


In [ ]:
from tqdm.auto import tqdm
import time

summaries_mistral = []
summaries_deepseek = []
#summaries_llama = []


for _, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
    text = str(row[TEXT_COLUMN])
    # Mistral
    try:
        summaries_mistral.append(summarize_mistral(text))
    except Exception as e:
        print("Mistral error:", e)
        summaries_mistral.append(-"")

    # DeepSeek
    try:
        summaries_deepseek.append(summarize_deepseek(text))
    except Exception as e:
        print("DeepSeek error:", e)
        summaries_deepseek.append("")

    # Llama
    #try:
    #    summaries_llama.append(summarize_llama(text))
    #except Exception as e:
    #    print("Llama error:", e)
    #   summaries_llama.append("")

In [ ]:
df_subset["summary_mistral"]  = summaries_mistral
df_subset["summary_deepseek"] = summaries_deepseek
#df_subset["summary_llama"] = summaries_llama

df_subset[[ID_COLUMN, "summary_mistral", "summary_deepseek"]].head()
#df_subset[[ID_COLUMN, "summary_mistral", "summary_deepseek", "summary_llama"]].head()

In [ ]:
output_path = "adneen_test_summaries_subset_250.csv"
df_subset.to_csv(output_path, index=False)
output_path


In [ ]:
!pip install -q bert-score

In [ ]:
from transformers import pipeline
from bert_score import score as bertscore
import torch
import re
import numpy as np

# QA model to answer questions from source and summary
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    tokenizer="deepset/roberta-base-squad2",
    device=0 if device == "cuda" else -1,
)

def generate_questions_from_summary(summary_text: str, k: int = 5) -> list:
    """
    Generate k factual questions conditioned on the summary using Mistral.
    This follows the QAGS idea: questions come from the summary itself.
    """
    summary_text = str(summary_text)

    prompt = (
        f"Write {k} short, factual questions (no answers) that test the key claims "
        f"in the following medical research summary. Number them as '1.', '2.', etc.\n\n"
        f"SUMMARY:\n{summary_text}\n\nQuestions:\n1."
    )

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(mistral_model.device)

    with torch.no_grad():
        output_ids = mistral_model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=256,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )

    text = mistral_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Keep only the part after "Questions:" to reduce junk
    if "Questions:" in text:
        text = text.split("Questions:", 1)[-1]

    questions = []
    for line in text.splitlines():
        line = line.strip()
        # matches "1. ..." or "2) ..." etc.
        if re.match(r"^\d+[\.\)]\s+", line):
            q = re.sub(r"^\d+[\.\)]\s+", "", line).strip()
            if q:
                questions.append(q)

    # Fallback: if parsing failed, split on '?'
    if not questions:
        for chunk in text.split("?"):
            q = chunk.strip()
            if q:
                questions.append(q + "?")

    return questions[:k]

In [ ]:
def answer_question(context: str, question: str) -> str:
    """
    Answer a question given a context using the QA pipeline.
    """
    try:
        result = qa_pipeline(question=question, context=context)
        return result.get("answer", "").strip()
    except Exception as e:
        print("QA error:", e)
        return ""

def qags_score_for_pair(source_text: str,
                        summary_text: str,
                        k: int = 5) -> float:
    """
    Approximate QAGS score for a single (source, summary) pair:
      1) Generate K questions from the summary
      2) Answer each question from the source and from the summary
      3) Compare answers using BERTScore F1
      4) Return the mean F1 as the factual consistency score
    """
    source_text = str(source_text)
    summary_text = str(summary_text)

    questions = generate_questions_from_summary(summary_text, k=k)
    if not questions:
        return 0.0

    src_answers = []
    sum_answers = []

    for q in questions:
        a_src = answer_question(source_text, q)
        a_sum = answer_question(summary_text, q)

        if not a_src or not a_sum:
            continue

        src_answers.append(a_src)
        sum_answers.append(a_sum)

    if not src_answers:
        return 0.0

    # BERTScore between summary answers and source answers
    P, R, F1 = bertscore(sum_answers, src_answers, lang="en",
                         rescale_with_baseline=True)
    return float(F1.mean().item())

In [ ]:
def compute_qags_for_model(df,
                           summary_col: str,
                           k_questions: int = 5,
                           max_samples: int = 15) -> list:
    """
    Compute QAGS-style scores for a given summary column
    on the first max_samples rows of df.
    """
    scores = []

    for idx, row in df.head(max_samples).iterrows():
        summary = row.get(summary_col, "")
        source  = row[TEXT_COLUMN]

        if not isinstance(summary, str) or not summary.strip():
            scores.append(0.0)
            continue

        s = qags_score_for_pair(source, summary, k=k_questions)
        print(f"Row {idx} – {summary_col} QAGS ≈ {s:.3f}")
        scores.append(s)

    return scores

# Compute QAGS-style scores for Mistral and DeepSeek
df_subset["qags_mistral"]  = compute_qags_for_model(df_subset, "summary_mistral")
df_subset["qags_deepseek"] = compute_qags_for_model(df_subset, "summary_deepseek")
#df_subset["qags_llama"] = compute_qags_for_model(df_subset, "summary_llama")

In [ ]:
def compute_qags_for_model(df,
                           summary_col: str,
                           k_questions: int = 5,
                           max_samples: int = 15):
    """
    Compute QAGS-style scores for a given summary column
    on the first max_samples rows of df.
    """
    scores = pd.Series(index=df.index, dtype=float)

    for idx, row in df.head(max_samples).iterrows():
        summary = row.get(summary_col, "")
        source  = row[TEXT_COLUMN]

        if not isinstance(summary, str) or not summary.strip():
            scores.loc[idx] = 0.0
            continue

        s = qags_score_for_pair(source, summary, k=k_questions)
        print(f"Row {idx} – {summary_col} QAGS ≈ {s:.3f}")
        scores.loc[idx] = s

    return scores
# Compute QAGS-style scores for Mistral and DeepSeek
df_subset["qags_mistral"]  = compute_qags_for_model(df_subset, "summary_mistral")
df_subset["qags_deepseek"] = compute_qags_for_model(df_subset, "summary_deepseek")



Rewritten to include all n=250 examples

In [ ]:
def compute_qags_for_model(
    df: pd.DataFrame,
    summary_col: str,
    text_col: str = TEXT_COLUMN,
    k_questions: int = 5,
):
    """
    Compute QAGS-style factual consistency scores for a summary column.

    QAGS is computed for every row in `df`.
    Rows with missing or empty summaries receive a score of 0.0.

    Returns
    -------
    pd.Series
        Index-aligned QAGS scores for all rows in df.
    """
    scores = pd.Series(index=df.index, dtype=float)

    for idx, row in df.iterrows():
        summary = row.get(summary_col)
        source  = row.get(text_col)

        if not isinstance(summary, str) or not summary.strip():
            scores.loc[idx] = 0.0
            continue

        score = qags_score_for_pair(
            source_text=source,
            summary_text=summary,
            k=k_questions
        )

        scores.loc[idx] = score
        print(f"Row {idx} | {summary_col} | QAGS ≈ {score:.3f}")

    return scores


In [ ]:
df_subset["qags_mistral"] = compute_qags_for_model(
    df_subset,
    summary_col="summary_mistral"
)

df_subset["qags_deepseek"] = compute_qags_for_model(
    df_subset,
    summary_col="summary_deepseek"
)


In [ ]:
import matplotlib.pyplot as plt

avg_qags = {
    "Mistral-7B":  df_subset["qags_mistral"].replace(0, np.nan).mean(),
    "DeepSeek-7B": df_subset["qags_deepseek"].replace(0, np.nan).mean(),
    #"Llama-8B": df_subset["qags_llama"].replace(0, np.nan).mean(),
}

models = list(avg_qags.keys())
values = [avg_qags[m] for m in models]

plt.figure(figsize=(6, 4))
plt.bar(models, values)
plt.ylabel("Average QAGS-style factual consistency (F1)")
plt.title("Factual consistency of side-effect summaries (QAGS-style)")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

avg_qags

Hallucination Rate Questions


Hallucination Rate = (# unanswerable questions from source) / (total questions)

In [ ]:
def is_unanswerable(answer: str, min_len: int = 3) -> bool:
    """
    Determine whether an answer is unanswerable.
    We treat empty or very short answers as unanswerable.
    """
    if not isinstance(answer, str):
        return True
    answer = answer.strip()
    return len(answer) < min_len


In [ ]:
def hallucination_rate_for_pair(
    source_text: str,
    summary_text: str,
    k: int = 5
) -> float:
    """
    Compute hallucination rate as:
    (# unanswerable questions from source) / (total questions)
    """
    questions = generate_questions_from_summary(summary_text, k=k)

    if not questions:
        return 0.0  # no questions → no detected hallucinations

    unanswerable_count = 0

    for q in questions:
        answer_from_source = answer_question(source_text, q)

        if is_unanswerable(answer_from_source):
            unanswerable_count += 1

    return unanswerable_count / len(questions)


In [ ]:
def compute_hallucination_rate_for_model(
    df,
    summary_col: str,
    k_questions: int = 5,
    max_samples: int = 15
) -> list:
    """
    Compute hallucination rates for a given model's summaries.
    """
    rates = []

    for idx, row in df.head(max_samples).iterrows():
        summary = row.get(summary_col, "")
        source = row[TEXT_COLUMN]

        if not isinstance(summary, str) or not summary.strip():
            rates.append(1.0)  # empty summary = fully hallucinated
            continue

        rate = hallucination_rate_for_pair(
            source,
            summary,
            k=k_questions
        )

        print(f"Row {idx} – {summary_col} Hallucination Rate ≈ {rate:.2f}")
        rates.append(rate)

    return rates


In [ ]:
def compute_hallucination_rate_for_model(
    df,
    summary_col: str,
    k_questions: int = 5,
    max_samples: int = 15
) -> list:
    """
    Compute hallucination rates for a given model's summaries.
    """
    rates = []

    for idx, row in df.head(max_samples).iterrows():
        summary = row.get(summary_col, "")
        source = row[TEXT_COLUMN]

        if not isinstance(summary, str) or not summary.strip():
            rates.append(1.0)  # empty summary = fully hallucinated
            continue

        rate = hallucination_rate_for_pair(
            source,
            summary,
            k=k_questions
        )

        print(f"Row {idx} – {summary_col} Hallucination Rate ≈ {rate:.2f}")
        rates.append(rate)

    return rates


In [ ]:
import pandas as pd

df = pd.read_csv("adneen_test_summaries_subset_250.csv")


In [ ]:
print(df.columns.tolist())


In [ ]:
mistral_col = "summary_mistral"

if "summary_deepseek" in df.columns:
    deepseek_col = "summary_deepseek"
elif "summar_deepseek" in df.columns:
    deepseek_col = "summar_deepseek"
else:
    raise ValueError("DeepSeek summary column not found")

df_summaries = df[[mistral_col, deepseek_col]]


In [ ]:
df["hallucination_mistral"] = compute_hallucination_rate_for_model(
    df,
    summary_col="summary_mistral"
)

df["hallucination_deepseek"] = compute_hallucination_rate_for_model(
    df,
    summary_col=deepseek_col
)
